# CPM2 cyclic-peptide binder design - Colab edition

This notebook runs the **CPM2 / CPNext** pipeline end-to-end on a single free
Colab GPU. CPM2 designs **cyclic-peptide inhibitors** of a protein-protein
interaction (PPI) in three stages:

```
  Stage 1                    Stage 2                     Stage 3
  cPEPmatch        ->        Boltz-2          ->         ProteinHunter
  (backbone match)           (fold-validate)             (sequence refine)

  scan a database of         re-fold each match as a     redesign the peptide
  ~400 cyclic peptides       peptide:target complex;     sequence (LigandMPNN)
  for a backbone that        keep the ones that fold     over several cycles to
  mimics the natural         into the intended pose      maximise predicted
  binding interface          (high ipTM, low RMSD)       interface confidence
```

**You give it:** one PDB of a protein-protein complex, and which chain is the
*target* (the protein you want to inhibit) and which is the *ligand* (the
interface the cyclic peptide should mimic).

**You get back:** a ranked table of designed cyclic-peptide sequences with
confidence scores, the predicted complex structures (PDB files), and a set of
triage plots.

---

### Before you start: turn on the GPU

`Runtime` -> `Change runtime type` -> **Hardware accelerator: T4 GPU** -> Save.

Without a GPU, Stages 2 and 3 are unusably slow.

### What to expect on the free tier

| | Free Colab (T4) |
|---|---|
| GPU | NVIDIA T4, ~15 GB VRAM |
| RAM | ~12 GB |
| Disk | ~78-110 GB (varies) |
| Max session | ~12 h; disconnects after ~90 min idle |

The default config below is deliberately small (one well-studied target,
a handful of peptide matches, a few designs each) so a full run fits inside
**roughly 1-2 hours** of T4 time. The one-time environment setup (conda +
Boltz weights download, ~8 GB) adds **~15-25 minutes** on the first run; mount
Google Drive (optional cell below) to cache it and skip that on re-runs.

> This notebook was authored without access to a live Colab runtime. Cells that
> could not be executed here are marked **[VERIFY ON COLAB]**. Read those notes.


## 1. Environment setup

### Why this is the tricky part (read this once)

The three stages have **conflicting dependencies**, which is why the normal
CPM2 install puts each stage in its own conda environment. The hard blocker for
a single pip environment is **Stage 1 (cPEPmatch)**:

* cPEPmatch's side-chain mutation step imports **Modeller** (`from modeller
  import *`) and **vmd-python** (`import vmd`). Modeller is distributed
  **only through conda** (the `salilab` channel) and needs a free academic
  **license key**; it is not pip-installable. vmd-python is likewise a
  conda-forge package.
* Stages 2 and 3 (Boltz-2, LigandMPNN/ProteinHunter) *are* pip-installable
  (`pip install boltz[cuda] ligandmpnn`) and need Python 3.10-3.12 + a CUDA
  build of torch, which Colab already has.

So a pure `pip`-only single environment is **not feasible** because of Modeller.

### The strategy this notebook uses: `condacolab` + one shared env

We install [`condacolab`](https://github.com/conda-incubator/condacolab), which
swaps Colab's Python for a Miniconda/Mambaforge base **in the same single
runtime**. Then:

1. `conda install -c salilab -c conda-forge modeller vmd-python` (Stage 1's
   blockers), supplying your Modeller license key.
2. `pip install boltz[cuda] ligandmpnn` and the light orchestration deps into
   that **same** base env (Stages 2 + 3).

Everything then runs in **one** Python env, so we call the existing CPM2 runner
functions with `conda_env=None` (Boltz, ProteinHunter) and run cPEPmatch's
script directly in the base env. We do **not** reimplement the pipeline and do
**not** modify `src/`.

> **Trade-off:** `condacolab.install()` **restarts the kernel once** (expected,
> not an error). After it restarts, just continue from the next cell.

> **Need a Modeller license?** It is free for academics and arrives by email
> within seconds: <https://salilab.org/modeller/registration.html>. Paste the
> key into `MODELLER_LICENSE` in the config cell below.


### 1a. Install condacolab  **[VERIFY ON COLAB]**

This installs Miniconda into the Colab runtime and **restarts the kernel
once**. That restart is normal. After the kernel comes back, **do not re-run
this cell** - just run the cells below it.

If you are *not* on Colab (e.g. testing locally in an env that already has
conda), skip this cell.


In [ ]:
# [VERIFY ON COLAB] Installs Miniconda into the runtime; restarts the kernel ONCE.
try:
    import condacolab
    condacolab.check()           # raises if conda isn't ready yet
    print("condacolab already installed and healthy.")
except Exception:
    !pip -q install condacolab
    import condacolab
    condacolab.install()         # <-- kernel restarts here; this is expected
    # Execution stops here on first run. Re-running cells below continues setup.


## 2. Configuration

Everything you might want to change lives in the single `CFG` dict below. The
scientific defaults are copied from the repo's `configs/mdm2_p53_v1.yaml`
(the MDM2/p53 benchmark agreed in the project's 2026-05-07 parameter audit),
trimmed for free-tier compute. Each field is commented with what it does and
why the default was chosen.

The default target is **MDM2 / p53 (PDB 1YCR)** - a small (~85-residue) target
with a single helical peptide in a well-defined pocket, the canonical PPI for
cyclic-peptide design. It is the smallest, best-characterised input in the
repo, so it is the safest first run.


In [ ]:
from pathlib import Path

CFG = {
    # ---- Modeller license (Stage 1 needs this; free academic key by email) ----
    # https://salilab.org/modeller/registration.html
    "MODELLER_LICENSE": "MODELIRANJE",   # <-- REPLACE with your real key

    # ---- Where the repo + caches live -------------------------------------
    "repo_url":  "https://github.com/your-org/cpepmatch2.git",  # <-- set to the real clone URL
    "repo_dir":  "/content/cpepmatch2",
    "use_drive": True,    # mount Google Drive to cache Boltz weights + MSA + the run
    "drive_cache": "/content/drive/MyDrive/cpm2_cache",

    # ---- Stage 0: which complex, which chains -----------------------------
    # The target is the protein you want to inhibit; the ligand is the natural
    # binder whose interface the cyclic peptide will mimic.
    "complex_pdb":        "data/input/1ycr.pdb",  # repo-relative or absolute
    "input_target_chain": "A",   # MDM2 N-terminal domain (the target)
    "input_ligand_chain": "B",   # p53 TAD peptide (the interface to mimic)

    # ---- Stage 1: cPEPmatch (backbone matching) ---------------------------
    "cpepmatch": {
        # Run knobs (passed to the cPEPmatch CLI)
        "motif_size": 4,            # # of CA atoms matched per motif (4-7). 4 = more, looser hits.
        "consecutive": True,         # match consecutive backbone stretches (vs scattered)
        "interface_cutoff": 6,       # A; residues within this of the partner count as "interface"
        "frmsd_threshold": 0.7,      # A; cPEPmatch's internal Dist-RMSD gate (paper range 0.3-1.5)
        "cyclization_type": None,    # e.g. "head to tail" to restrict the DB; None = all topologies
        "exclude_non_standard": False,
        # Post-run filters (applied in this notebook)
        "fit_rmsd_threshold": 0.7,   # A; keep matches whose 3D superposition fit-RMSD <= this
        "min_residues": 6,           # drop peptides shorter than this
        "max_residues": 22,          # ...or longer than this
        "unique_sources": True,      # one match per source peptide (best fit) -> cheaper, diverse
        "mutated_only": True,        # only keep matches Modeller could mutate (drops exotic residues)
        # FREE-TIER GUARDRAIL: hard cap on how many matches advance to Boltz.
        # Each match costs one Boltz fold + (if it passes) one ProteinHunter run.
        "max_matches": 4,            # keep only the 4 best-fit matches. Raise cautiously.
    },

    # ---- Stage 2: Boltz-2 (fold validation) -------------------------------
    "boltz": {
        "step_scale": 1.5,          # Boltz-2 native diffusion temperature default
        "diffusion_samples": 1,     # FREE-TIER: 1 sample/match (repo uses 5; 1 is much cheaper)
        "recycling_steps": 3,       # FREE-TIER: 3 (repo uses 6; 3 is the documented cost/quality knee)
        "use_potentials": True,     # cleaner cyclic backbones (clash + ring-closure potentials)
        "seed": 42,                 # reproducible diffusion
        "output_format": "mmcif",   # Boltz-2 native; we read CIF downstream
        "use_msa_server": True,     # fetch MSA from the public ColabFold server (needs internet)
        # Cyclisation detection thresholds (distance-based, Angstrom)
        "cyclization_distance": 1.5,
        "disulfide_distance": 2.5,
        # Filters: a match "passes" Stage 2 only if it folds confidently AND
        # close to the cPEPmatch pose.
        "iptm_threshold": 0.7,      # min interface pTM (0-1)
        "plddt_threshold": 0.7,     # min complex pLDDT (0-1)
        "rmsd_threshold": 2.0,      # max CA-RMSD (A) of the predicted peptide vs the cPEPmatch scaffold
    },

    # ---- Stage 3: ProteinHunter (sequence refinement) ---------------------
    "proteinhunter": {
        "num_designs": 2,           # FREE-TIER: 2 parallel designs/match (repo uses 4)
        "num_cycles": 5,            # FREE-TIER: 5 redesign cycles (repo uses 10)
        "diffusion_samples": 1,     # FREE-TIER: 1 (repo uses 3; >1 reduces per-cycle ipTM noise)
        "recycling_steps": 3,       # match Stage 2
        "seed": 42,
        "temperature": 0.1,         # LigandMPNN sampling temperature (low = conservative)
        "omit_AA": "C",             # forbid free cysteines in redesign (avoids spurious disulfides)
        "template_force": True,     # Boltz pulls the target toward the input template during refine
        "template_force_threshold": 2.0,  # A; how far the target may drift under forcing
        "iptm_threshold": 0.7,      # report designs above this interface pTM
    },
}

# Echo the budget so the user sees the compute envelope before committing.
print("Run budget (free-tier sized):")
print(f"  Stage 1: keep up to {CFG['cpepmatch']['max_matches']} cPEPmatch matches")
print(f"  Stage 2: {CFG['boltz']['diffusion_samples']} Boltz sample(s) x up to "
      f"{CFG['cpepmatch']['max_matches']} matches")
print(f"  Stage 3: {CFG['proteinhunter']['num_designs']} designs x "
      f"{CFG['proteinhunter']['num_cycles']} cycles per validated match")
if CFG["MODELLER_LICENSE"] in ("", "MODELIRANJE"):
    print("\n[!] Set CFG['MODELLER_LICENSE'] to your real key before running Stage 1 setup.")


### 2a. (Optional) Mount Google Drive for caching  **[VERIFY ON COLAB]**

Boltz downloads ~8 GB of model weights on first use, and the MSA fetch can take
a few minutes. Mounting Drive lets us cache both so **re-runs skip the slow
parts**. This is optional - if you skip it (or set `CFG["use_drive"] = False`),
everything still works as a one-shot run, you just pay the download cost each
fresh runtime.


In [ ]:
# [VERIFY ON COLAB] Mounts Drive and points the Boltz weight cache + MSA cache at it.
import os
from pathlib import Path

BOLTZ_CACHE = Path.home() / ".boltz"   # default; overridden below if Drive is on
DRIVE_OK = False

if CFG["use_drive"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        cache_root = Path(CFG["drive_cache"])
        cache_root.mkdir(parents=True, exist_ok=True)
        BOLTZ_CACHE = cache_root / "boltz_weights"
        BOLTZ_CACHE.mkdir(parents=True, exist_ok=True)
        # Symlink ~/.boltz -> Drive so Boltz finds cached weights automatically.
        home_boltz = Path.home() / ".boltz"
        if home_boltz.is_symlink() or home_boltz.exists():
            if not home_boltz.is_symlink():
                print(f"~/.boltz already exists as a real dir; leaving it. "
                      f"Weights will NOT be cached to Drive.")
                BOLTZ_CACHE = home_boltz
        else:
            home_boltz.symlink_to(BOLTZ_CACHE)
        DRIVE_OK = True
        print(f"Drive mounted. Boltz weights cache -> {BOLTZ_CACHE}")
    except Exception as e:
        print(f"Drive mount failed ({e}); continuing without caching (one-shot run).")
else:
    print("use_drive=False; running one-shot (no weight caching).")


## 3. Get the CPM2 code and the vendored tools

We need two things on disk:

1. **The CPM2 repo** (`src/`, `configs/`, `data/input/`). Set `CFG["repo_url"]`
   to wherever you cloned it (your fork / lab GitHub). If you instead uploaded
   the repo as a zip, adapt the clone cell to unzip it into `CFG["repo_dir"]`.
2. **The vendored external tools** under `lib/`: `cPEPmatch/`, `Protein-Hunter/`
   (which bundles `LigandMPNN/`), and the Boltz fork. These are **gitignored**
   in the repo (see `lib/VERSIONS.md`), so they are *not* pulled by the clone -
   we fetch them from their upstream sources at the pinned versions recorded in
   `lib/VERSIONS.md`:
   * cPEPmatch - github.com/briandasantini/cPEPmatch @ `e3c7469`
   * Protein-Hunter - github.com/yehlincho/Protein-Hunter
   * LigandMPNN - github.com/dauparas/LigandMPNN (lives inside Protein-Hunter)

> The exact `lib/` layout the runners expect: `lib/cPEPmatch/cpepmatch.py`,
> `lib/Protein-Hunter/`, `lib/Protein-Hunter/LigandMPNN/run.py`. If you already
> have a populated `lib/` (e.g. you cloned a tarball of the full project),
> set `CFG["repo_url"]` to that and this cell will detect it and skip the
> upstream fetch.


In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO = Path(CFG["repo_dir"])

def sh(cmd, **kw):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")

# --- 1. Repo ---
if not (REPO / "src").is_dir():
    sh(f"git clone --depth 1 {CFG['repo_url']} {REPO}")
else:
    print(f"Repo already present at {REPO}")

LIB = REPO / "lib"
LIB.mkdir(exist_ok=True)

# --- 2. Vendored tools (only if missing) ---
if not (LIB / "cPEPmatch" / "cpepmatch.py").exists():
    sh(f"git clone https://github.com/briandasantini/cPEPmatch.git {LIB/'cPEPmatch'}")
    # Pin to the version recorded in lib/VERSIONS.md.
    sh(f"git -C {LIB/'cPEPmatch'} checkout e3c746999afa0715d819c42f550d8c4ccae48489 || true")
else:
    print("lib/cPEPmatch present")

if not (LIB / "Protein-Hunter").is_dir():
    sh(f"git clone https://github.com/yehlincho/Protein-Hunter.git {LIB/'Protein-Hunter'}")
else:
    print("lib/Protein-Hunter present")

# LigandMPNN nests inside Protein-Hunter; clone if the fork didn't bundle it.
if not (LIB / "Protein-Hunter" / "LigandMPNN" / "run.py").exists():
    sh(f"git clone https://github.com/dauparas/LigandMPNN.git {LIB/'Protein-Hunter'/'LigandMPNN'}")
else:
    print("lib/Protein-Hunter/LigandMPNN present")

os.chdir(REPO)
print(f"\nWorking directory: {os.getcwd()}")


## 4. Install dependencies into the one shared env  **[VERIFY ON COLAB]**

Two installers, one environment:

* **conda** (via condacolab's base): `modeller` + `vmd-python` for Stage 1.
  The Modeller license key from `CFG` is passed via the `KEY_MODELLER` env var
  so the install is non-interactive.
* **pip**: `boltz[cuda]` (Stage 2) and `ligandmpnn` (Stage 3) plus the light
  orchestration deps the CPM2 `src/` package imports (biopython, gemmi,
  pandas, matplotlib, plotly, pyyaml, MDAnalysis, py3Dmol).

This is the slow cell (~10-20 min first run). It is idempotent: re-running
re-checks and skips what's present.


In [ ]:
# [VERIFY ON COLAB] conda + pip install into the shared base env.
import os, importlib, subprocess

os.environ["KEY_MODELLER"] = CFG["MODELLER_LICENSE"]
# The salilab conda package bakes the key into modeller.key at INSTALL time,
# so a placeholder here installs a broken Modeller and re-running this cell
# will NOT fix it (have('modeller') stays True, the install is skipped).
# Fail loudly now instead of silently at Stage 1.
if CFG["MODELLER_LICENSE"] in ("", "MODELIRANJE"):
    raise ValueError(
        "Set CFG['MODELLER_LICENSE'] to your free academic Modeller key in "
        "the Configuration cell before running setup.")

def have(mod):
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False

# --- conda: Modeller + vmd-python (Stage 1 blockers) ---
if not have("modeller"):
    print("Installing Modeller + vmd-python via conda (salilab + conda-forge)...")
    # condacolab exposes `conda`/`mamba` on PATH after its kernel restart.
    sh("mamba install -y -c salilab -c conda-forge modeller vmd-python || "
       "conda install -y -c salilab -c conda-forge modeller vmd-python")
else:
    print("modeller already importable")

# --- pip: Boltz-2 + LigandMPNN + light orchestration deps ---
if not have("boltz"):
    print("Installing Boltz-2 (GPU) ...")
    sh("pip -q install 'boltz[cuda]'")
else:
    print("boltz already importable")

if not have("ligandmpnn"):
    # ProteinHunter imports LigandMPNN from lib/, but the pip package pulls the
    # right torch-compatible deps and ships the model weights; install both.
    print("Installing ligandmpnn ...")
    sh("pip -q install ligandmpnn || echo 'ligandmpnn pip install failed; lib/ copy will be used'")

# Light orchestration deps used by cpm2.{utils,analysis,boltz,filters}.
sh("pip -q install biopython gemmi pandas matplotlib plotly pyyaml MDAnalysis py3Dmol seaborn")

# Make the cpm2 package importable WITHOUT touching pyproject / editable install.
import sys
if str((REPO / 'src')) not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

print("\nVerifying imports:")
for m in ["modeller", "boltz", "torch", "Bio", "gemmi", "pandas", "matplotlib"]:
    print(f"  {m:12s}: {'OK' if have(m) else 'MISSING'}")
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}  "
      f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


### 4a. Download LigandMPNN model weights  **[VERIFY ON COLAB]**

ProteinHunter's redesign step loads LigandMPNN checkpoints from
`lib/Protein-Hunter/LigandMPNN/model_params/`. They are a few hundred MB and
are fetched once from the UW IPD server. (Boltz-2's much larger ~8 GB weights
download automatically on its first prediction, into `~/.boltz`, which we
pointed at Drive above.)


In [ ]:
# [VERIFY ON COLAB] Fetch LigandMPNN checkpoints (a few hundred MB).
from pathlib import Path
mp = REPO / "lib" / "Protein-Hunter" / "LigandMPNN" / "model_params"
if mp.is_dir() and any(mp.glob("*.pt")):
    print(f"LigandMPNN weights already present in {mp}")
else:
    script = REPO / "lib" / "Protein-Hunter" / "LigandMPNN" / "get_model_params.sh"
    if script.exists():
        sh(f"bash {script} {mp}")
    else:
        print(f"get_model_params.sh not found at {script}; "
              "check lib/Protein-Hunter/LigandMPNN was cloned correctly.")


## 5. Build the pipeline config and the per-run directory

We assemble the config the CPM2 runners expect and create a `data/runs/<run_id>/`
directory in the exact layout `cpm2.analysis.load_run` reads for the plots at
the end. `run_id = colab_<timestamp>`.


In [ ]:
import os, time, yaml
from pathlib import Path

os.chdir(REPO)
PIPELINE_ROOT = REPO

# Resolve the input complex path (repo-relative or absolute).
complex_pdb = Path(CFG["complex_pdb"])
if not complex_pdb.is_absolute():
    complex_pdb = (PIPELINE_ROOT / complex_pdb).resolve()
assert complex_pdb.exists(), f"input complex not found: {complex_pdb}"

# Per-run directory in the canonical CPM2 layout.
RUN_ID = f"colab_{time.strftime('%Y%m%d_%H%M%S')}"
RUN_ROOT = PIPELINE_ROOT / "data" / "runs" / RUN_ID
INTERMEDIATE = RUN_ROOT / "intermediate"
OUTPUT = RUN_ROOT / "output"
for d in (INTERMEDIATE, OUTPUT):
    d.mkdir(parents=True, exist_ok=True)

# A plain config dict for the runners (mirrors configs/*.yaml + default_config).
config = {
    "complex_pdb": complex_pdb,
    "input_ligand_chain": CFG["input_ligand_chain"],
    "input_target_chain": CFG["input_target_chain"],
    "cpepmatch": dict(CFG["cpepmatch"]),
    "boltz": dict(CFG["boltz"]),
    "proteinhunter": dict(CFG["proteinhunter"]),
}

# Persist the config next to the run so analysis.load_run can pick it up, and
# write a minimal manifest (load_run reads it but tolerates absence).
(RUN_ROOT / "config.yaml").write_text(
    yaml.safe_dump({k: (str(v) if isinstance(v, Path) else v) for k, v in config.items()})
)
import json
(RUN_ROOT / "manifest.json").write_text(json.dumps({
    "run_id": RUN_ID, "config_name": "colab",
    "timestamp_iso": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "config": {
        "proteinhunter": {"template_force": config["proteinhunter"]["template_force"],
                          "template_force_threshold": config["proteinhunter"]["template_force_threshold"]}
    },
}, indent=2))

print(f"run_id        = {RUN_ID}")
print(f"run dir       = {RUN_ROOT}")
print(f"input complex = {complex_pdb}")


## 6. Stage 0 - import and standardise the complex

We extract the two chains of interest and rename them to the pipeline's
convention: **L** = ligand (interface to mimic), **T** = target (what the
peptide must bind). A target-only `template.cif` is written for Stage 3.


In [ ]:
from cpm2.utils.pdb_utils import import_complex, get_chain_ids, get_sequence, extract_chain_to_cif

print(f"Available chains in input: {get_chain_ids(config['complex_pdb'])}")

stage0 = INTERMEDIATE / "0_import"
stage0.mkdir(parents=True, exist_ok=True)
processed_pdb = stage0 / "processed.pdb"
template_cif = stage0 / "template.cif"

import_complex(
    complex_pdb=config["complex_pdb"],
    ligand_chain=config["input_ligand_chain"],
    target_chain=config["input_target_chain"],
    output_pdb=processed_pdb,
)
extract_chain_to_cif(processed_pdb, "T", template_cif)

ligand_seq = get_sequence(processed_pdb, "L")
target_seq = get_sequence(processed_pdb, "T")
print(f"  ligand (L): {len(ligand_seq)} residues  {ligand_seq}")
print(f"  target (T): {len(target_seq)} residues")

# Stage 3 needs the target template path in its config.
config["proteinhunter"]["template_path"] = str(template_cif)


## 7. Stage 1 - cPEPmatch (backbone matching)

cPEPmatch scans its database of ~400 cyclic peptides for backbones that mimic
the ligand interface, mutates matching side chains (Modeller), and emits one
PDB per match. We then filter to the best few (the `max_matches` guardrail).

> **Why this stage looks different from Stages 2/3.** The Boltz and
> ProteinHunter runners accept `conda_env=None` to skip the `conda run -n <env>`
> wrapper and run in the current env - exactly the Colab case. The cPEPmatch
> runner (`cpepmatch.run()`) does **not** have that switch; it always prepends
> `conda run -n <env>`. Since we are not allowed to modify `src/`, we run
> cPEPmatch's CLI directly in the shared env here (the same command the runner
> would build, minus the `conda run` prefix) and then reuse the runner's own
> `parse_match_list()` to parse the output into `CPEPmatchResult` objects. No
> reimplementation, no `src/` edits.


In [ ]:
import subprocess
from cpm2.runners import cpepmatch as cpepmatch_runner   # reuse parse_match_list()
from cpm2.filters import cpepmatch_filters

stage1 = INTERMEDIATE / "1_cpepmatch"
if stage1.exists():
    import shutil; shutil.rmtree(stage1)
stage1.mkdir(parents=True, exist_ok=True)

lib_cpepmatch = REPO / "lib" / "cPEPmatch"
db_location = lib_cpepmatch / "database"
cpepmatch_script = lib_cpepmatch / "cpepmatch.py"

# cPEPmatch expects the input PDB in its working dir.
import shutil
shutil.copy(processed_pdb, stage1 / processed_pdb.name)

cp = config["cpepmatch"]
# Build the bare CLI command (identical to cpepmatch.run() minus `conda run`).
cmd = [
    "python", str(cpepmatch_script),
    "-n", processed_pdb.stem,
    "-p", "L", "-t", "T",
    "-wl", str(stage1) + "/",
    "-dl", str(db_location) + "/",
    "-ms", str(cp["motif_size"]),
    "-cs", str(cp["consecutive"]),
    "-ic", str(cp["interface_cutoff"]),
    "-ft", str(cp["frmsd_threshold"]),
]
if cp.get("cyclization_type"):
    cmd += ["-ct", cp["cyclization_type"]]
if cp.get("exclude_non_standard"):
    cmd += ["-ens", "True"]

print("$ " + " ".join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True, cwd=str(stage1))
if proc.returncode != 0:
    raise RuntimeError(
        "Stage 1 (cPEPmatch) failed. Most common causes on Colab:\n"
        "  - Modeller license not set / invalid (check CFG['MODELLER_LICENSE'])\n"
        "  - vmd-python missing (re-run the conda install cell)\n"
        f"--- stdout ---\n{proc.stdout[-2000:]}\n--- stderr ---\n{proc.stderr[-2000:]}"
    )

# Parse with the runner's own parser (also cross-checks residue counts).
matches = cpepmatch_runner.parse_match_list(
    stage1 / "match_list.txt", stage1, database_location=db_location
)
print(f"cPEPmatch returned {len(matches)} raw matches")

# Filter, then apply the free-tier cap on how many advance to Boltz.
filtered = cpepmatch_filters.apply_all_filters(
    matches,
    min_residues=config["cpepmatch"]["min_residues"],
    max_residues=config["cpepmatch"]["max_residues"],
    unique_sources=config["cpepmatch"]["unique_sources"],
    mutated_only=config["cpepmatch"]["mutated_only"],
    max_fit_rmsd=config["cpepmatch"]["fit_rmsd_threshold"],
)
cap = config["cpepmatch"]["max_matches"]
filtered = filtered[:cap]
print(f"{len(filtered)} matches kept (capped at {cap}):")
for m in filtered:
    print(f"  {m.name}: fit_rmsd={m.fit_rmsd:.2f} A, {m.num_residues} residues")
if not filtered:
    print("\n[!] No matches survived filtering. Loosen cpepmatch.frmsd_threshold / "
          "fit_rmsd_threshold or motif_size and re-run Stage 1.")


### 7a. Rename the peptide chain to P (Boltz convention)

In [ ]:
from cpm2.utils.pdb_utils import rename_chain
from Bio.PDB import PDBParser as _PDBParser

renamed_dir = INTERMEDIATE / "1_cpepmatch_renamed"
renamed_dir.mkdir(parents=True, exist_ok=True)
for old in renamed_dir.glob("*.pdb"):
    old.unlink()

def _peptide_chain(pdb_path, assumed):
    # cPEPmatch match PDBs usually carry the peptide in chain A (Mutated) or B
    # (some NotMutated), but several NotMutated files have ONLY chain A. Detect
    # the chains actually present so a missing assumed-chain never hard-crashes
    # rename_chain; keep the assumed chain when it exists (preserves behaviour).
    chains = [c.id for c in
              _PDBParser(QUIET=True).get_structure("x", str(pdb_path))[0]]
    if assumed in chains:
        return assumed
    return chains[0] if chains else assumed

for match in filtered:
    fname = match.pdb_path.name
    assumed = "B" if "NotMutated" in fname else "A"
    src = stage1 / fname
    chosen = _peptide_chain(src, assumed)
    dst = renamed_dir / fname
    rename_chain(src, chosen, "P", dst)
    match.pdb_path = dst
print(f"Renamed peptide chain -> P for {len(filtered)} matches in {renamed_dir}")


## 8. Stage 2 - Boltz-2 fold validation

For each match we build a Boltz YAML (peptide chain P + target chain T, with any
detected cyclisation / disulfide bonds), then fold the complex. A match
**passes** if it folds confidently (ipTM, pLDDT) into a pose close to the
cPEPmatch scaffold (low CA-RMSD).

> **First Boltz call downloads ~8 GB of weights** to `~/.boltz` (cached to Drive
> if you mounted it). Expect several minutes before the first prediction starts.
> The MSA is fetched from the public ColabFold server (needs internet).


In [ ]:
from cpm2.utils.pdb_utils import get_sequence, get_modifications
from cpm2.utils.constraints import detect_constraints_from_pdb
from cpm2.boltz import build_boltz_yaml, write_boltz_yaml

yaml_dir = INTERMEDIATE / "2_boltz" / "yaml_input"
if yaml_dir.exists():
    import shutil; shutil.rmtree(yaml_dir)
yaml_dir.mkdir(parents=True, exist_ok=True)

cyc = config["boltz"]["cyclization_distance"]
ss = config["boltz"]["disulfide_distance"]

# Target constraints (distance-based, used for the YAML).
target_constraints = detect_constraints_from_pdb(processed_pdb, "T", cyc, ss)

match_pdbs = sorted(renamed_dir.glob("match*.pdb"))
for mp in match_pdbs:
    cp_seq = get_sequence(mp, "P")
    mods = get_modifications(mp, "P")
    cp_constraints = detect_constraints_from_pdb(mp, "P", cyc, ss)
    ydict = build_boltz_yaml(target_seq, cp_seq, mods, cp_constraints, target_constraints)
    write_boltz_yaml(yaml_dir / f"{mp.stem}.yaml", ydict)
    topo = "head-to-tail" if cp_constraints["head_to_tail"] else (
        "disulfide/other" if any(cp_constraints[k] for k in ("disulfides","lactams","thioethers")) else "linear")
    print(f"  {mp.stem}: {len(cp_seq)} aa, {topo}")
print(f"\nWrote {len(match_pdbs)} Boltz YAMLs to {yaml_dir}")


In [ ]:
from cpm2.runners.boltz_runner import BoltzPredictConfig
from cpm2.runners import boltz_runner as boltz
from cpm2.filters import boltz_filters

predict_config = BoltzPredictConfig.from_dict(config["boltz"])
# Route Boltz's weight cache at the (possibly Drive-backed) location.
predict_config.cache = BOLTZ_CACHE

boltz_out = INTERMEDIATE / "2_boltz" / "predictions"
pdb_map = {p.stem: p for p in match_pdbs}

try:
    results = boltz.run_batch(
        yaml_dir=yaml_dir,
        output_dir=boltz_out,
        conda_env=None,             # one shared env
        predict_config=predict_config,
        input_pdbs=pdb_map,
        cp_chain="P",
        batch_size=None,            # small set; fold them in one call
    )
except Exception as e:
    raise RuntimeError(
        "Stage 2 (Boltz-2) failed. Common Colab causes:\n"
        "  - GPU not enabled (Runtime -> Change runtime type -> T4 GPU)\n"
        "  - out of VRAM (lower diffusion_samples, or fewer matches)\n"
        "  - MSA server unreachable (transient; re-run)\n"
        f"Original error:\n{e}"
    )

print(f"\nFolded {len(results)} complexes:")
for r in results:
    rstr = f", RMSD={r.rmsd_to_input:.2f} A" if r.rmsd_to_input is not None else ""
    print(f"  {r.name}: ipTM={r.iptm:.3f} pLDDT={r.plddt:.2f}{rstr}")

validated = boltz_filters.apply_all_filters(
    results,
    iptm_threshold=config["boltz"]["iptm_threshold"],
    plddt_threshold=config["boltz"]["plddt_threshold"],
    rmsd_threshold=config["boltz"]["rmsd_threshold"],
)
print(f"\n{len(validated)} matches passed Stage 2 validation.")
if not validated:
    print("[!] None passed. Lower boltz.iptm_threshold / plddt_threshold or "
          "loosen rmsd_threshold, or accept that these matches don't fold well.")


## 9. Stage 3 - ProteinHunter sequence refinement

For each validated complex, ProteinHunter iteratively redesigns the peptide
sequence (LigandMPNN proposes mutations; Boltz re-folds and scores) for
`num_cycles` cycles, keeping the best design(s). This is the most GPU-intensive
stage: cost scales as `validated x num_designs x num_cycles`.


In [ ]:
import yaml as _yaml
from cpm2.runners import proteinhunter
from cpm2.runners.proteinhunter import ProteinHunterConfig

ph_config = ProteinHunterConfig.from_dict(config["proteinhunter"])
ph_out = INTERMEDIATE / "3_proteinhunter"
all_designs = []

for structure in validated:
    print(f"\n=== refining {structure.name} ===")
    # Read cyclic flag from the Stage-2 YAML.
    is_cyclic = False
    yp = yaml_dir / f"{structure.name}.yaml"
    if yp.exists():
        for entry in (_yaml.safe_load(yp.read_text()).get("sequences") or []):
            if "protein" in entry and entry["protein"].get("id") == "P":
                is_cyclic = entry["protein"].get("cyclic", False)
    try:
        designs = proteinhunter.run_refine(
            input_structure=structure.output_structure,
            cp_chain="P",
            target_chain="T",
            output_dir=ph_out / structure.name,
            config=ph_config,
            is_cyclic=is_cyclic,
            conda_env=None,          # one shared env
        )
        for d in designs:
            all_designs.append(d.to_dict())
            print(f"  design {d.design_num}: ipTM={d.iptm:.3f} pLDDT={d.plddt:.2f} "
                  f"(best cycle {d.cycle})")
        print(f"  {len(designs)} design(s) passed threshold")
    except Exception as e:
        print(f"  ERROR refining {structure.name}: {e}")
        continue

print(f"\nTotal designs: {len(all_designs)}")


## 10. Save results in the canonical layout

We write `output/summary.csv` so the standard `cpm2.analysis` loader and plots
work exactly as they do for headless runs.


In [ ]:
import pandas as pd

summary_df = pd.DataFrame(all_designs)
if len(summary_df):
    summary_df = summary_df.sort_values("iptm", ascending=False)
summary_path = OUTPUT / "summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Wrote {len(summary_df)} designs to {summary_path}")

# Optionally copy the whole run to Drive so it survives the runtime.
if DRIVE_OK:
    import shutil
    dst = Path(CFG["drive_cache"]) / "runs" / RUN_ID
    try:
        shutil.copytree(OUTPUT, dst / "output", dirs_exist_ok=True)
        shutil.copy(RUN_ROOT / "manifest.json", dst / "manifest.json")
        print(f"Copied results to Drive: {dst}")
    except Exception as e:
        print(f"(could not copy results to Drive: {e})")

summary_df.head(10)


## 11. Results - ranked designs and triage plots

These reuse the project's own `cpm2.analysis` functions (`load_run`,
`plot_topology`, `plot_distributions`, `plot_triage`, `plot_convergence`,
`rank`) so the output is identical to what the lab sees for a headless run.
Every plot is wrapped so an empty / partial run degrades to a friendly message
instead of crashing.


In [ ]:
import importlib
from cpm2 import analysis
importlib.reload(analysis)   # ensure we use the repo's current analysis module

rd = None
try:
    rd = analysis.load_run(RUN_ID)
    print(rd.overview())
except Exception as e:
    print(f"Could not load run for analysis: {e}")
    if not len(summary_df):
        print("(summary.csv is empty - no designs passed Stage 3. "
              "Loosen thresholds and re-run.)")


In [ ]:
# Ranked table of the best designs (top by ipTM).
if rd is not None and rd.n_designs:
    display(analysis.rank(rd, by="iptm", n=20))
else:
    print("No designs to rank.")


In [ ]:
# Cyclisation-topology breakdown.
import matplotlib.pyplot as plt
if rd is not None and rd.n_designs:
    try:
        analysis.plot_topology(rd)   # returns a matplotlib Figure
        plt.show()
    except Exception as e:
        print(f"plot_topology unavailable: {e}")
else:
    print("No designs to plot.")


In [ ]:
# Score & RMSD distributions across all designs.
if rd is not None and rd.n_designs:
    try:
        analysis.plot_distributions(rd); import matplotlib.pyplot as plt; plt.show()
    except Exception as e:
        print(f"plot_distributions unavailable: {e}")
else:
    print("No designs to plot.")


In [ ]:
# Triage plane: ipTM vs peptide drift (top-left = best). Static matplotlib.
if rd is not None and rd.n_designs:
    try:
        analysis.plot_triage(rd); import matplotlib.pyplot as plt; plt.show()
    except Exception as e:
        print(f"plot_triage unavailable: {e}")
else:
    print("No designs to plot.")


In [ ]:
# Per-cycle convergence of the top designs (target CA-RMSD over refinement cycles).
if rd is not None and rd.n_designs:
    try:
        analysis.plot_convergence(rd, n=6); import matplotlib.pyplot as plt; plt.show()
    except Exception as e:
        print(f"plot_convergence unavailable (per-cycle data may be absent): {e}")
else:
    print("No designs to plot.")


## 12. Where your results are

* **Ranked sequences + scores:** `data/runs/<run_id>/output/summary.csv`
* **Predicted complex PDBs:** under `data/runs/<run_id>/intermediate/3_proteinhunter/<match>/`
  (the `output_pdb` column in the table points at the best one per design).
* If you mounted Drive, a copy of `output/` is under
  `MyDrive/cpm2_cache/runs/<run_id>/`.

To design against **a different target**: change `complex_pdb`,
`input_target_chain`, `input_ligand_chain` in `CFG`, re-run from Section 5.
Keep `max_matches`, `num_designs`, and `num_cycles` small until you know the
run fits your time budget.
